# T034 - XGBoost GPU tuning on T026 wide request features\n\nTen XGBoost configurations (leaves/depth, eta, column sampling, bins, native categoricals, gradient-based sampling, regularisation) on the T029 wide feature set; original holdout, early stopping on the holdout, holdout predictions saved with file row ids for blending. GPU, no submission.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

sources = {'feature_screen.py': '"""Local LightGBM screen of request-context, train+test count and target-encoding features.\n\nTrains on rows before the original primary cutoff and scores the original\n307,528-row holdout. request_id is used only for grouping, never as an input.\n"""\nimport argparse\nfrom datetime import datetime\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\n\nCUTOFF = datetime.fromisoformat("2026-10-28T08:06:52.397000")\nUSER, ITEM = "feature5", "feature18"\nLOW_CATEGORICAL = [f"feature{i}" for i in [2, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 19, 21, 22, 25]]\nCOUNT_KEYS = [USER, ITEM, "feature23", "feature20", "feature19", "feature9", "feature12"]\nENCODED = [ITEM, "feature23", "feature20", "feature19", "feature9", "feature12", "feature21", USER]\nRANKED = ["feature418", "feature419", "feature411", "feature410", "feature430",\n          "feature431", "feature417", "feature352", "feature346", "feature480"]\nSMOOTHING = 20.0\n\n\ndef scalar_numeric(schema):\n    skip = {"feature1", "feature3"}\n    return [c for c, t in schema.items()\n            if c.startswith("feature") and c not in skip\n            and str(t) in {"Float32", "Int64", "Int8"}]\n\n\ndef time_seconds(column):\n    return pl.col(column).dt.epoch("ms").cast(pl.Float64) / 1000.0\n\n\ndef count_features(keys):\n    """Label-free counts over the complete train+test population."""\n    frame = keys.with_columns(time_seconds("feature1").alias("t"))\n    exprs = []\n    for key in COUNT_KEYS:\n        exprs += [pl.len().over(key).alias(f"cnt_{key}"),\n                  pl.col("request_id").n_unique().over(key).alias(f"req_{key}")]\n    frame = frame.with_columns(exprs)\n    pair = [USER, ITEM]\n    frame = frame.with_columns(\n        pl.len().over(pair).alias("cnt_user_item"),\n        pl.col("t").min().over(ITEM).alias("item_first"),\n        pl.col("t").max().over(ITEM).alias("item_last"),\n        pl.col("t").min().over(USER).alias("user_first"),\n        pl.col("t").max().over(USER).alias("user_last"),\n        pl.len().over([ITEM, pl.col("feature1").dt.truncate("1h")]).alias("cnt_item_hour"),\n        pl.len().over([ITEM, pl.col("feature1").dt.truncate("1d")]).alias("cnt_item_day"),\n    )\n    requests = (\n        frame.group_by("request_id").agg(pl.col(USER).first(), pl.col("t").first())\n        .sort([USER, "t", "request_id"])\n        .with_columns(\n            pl.int_range(pl.len()).over(USER).alias("user_req_index"),\n            (pl.col("t") - pl.col("t").shift(1).over(USER)).alias("user_prev_gap"),\n            (pl.col("t").shift(-1).over(USER) - pl.col("t")).alias("user_next_gap"),\n        )\n        .select("request_id", "user_req_index", "user_prev_gap", "user_next_gap")\n    )\n    frame = frame.join(requests, on="request_id", how="left")\n    pairs = (\n        frame.select(USER, ITEM, "t").unique().sort([USER, ITEM, "t"])\n        .with_columns(\n            pl.int_range(pl.len()).over([USER, ITEM]).alias("user_item_index"),\n            (pl.col("t") - pl.col("t").shift(1).over([USER, ITEM])).alias("user_item_prev_gap"),\n            (pl.col("t").shift(-1).over([USER, ITEM]) - pl.col("t")).alias("user_item_next_gap"),\n        )\n    )\n    frame = frame.join(pairs, on=[USER, ITEM, "t"], how="left")\n    frame = frame.with_columns(\n        (pl.col("req_feature5") - pl.col("user_req_index") - 1).alias("user_reqs_after"),\n        (pl.col("t") - pl.col("item_first")).alias("item_age"),\n        (pl.col("item_last") - pl.col("t")).alias("item_remaining"),\n        (pl.col("t") - pl.col("user_first")).alias("user_since_first"),\n        (pl.col("user_last") - pl.col("t")).alias("user_until_last"),\n    ).drop("item_first", "item_last", "user_first", "user_last")\n    names = [c for c in frame.columns if c not in keys.columns and c != "t"]\n    return frame.select("row", *names), names\n\n\ndef request_features(frame):\n    """Within-request context; frame is one split population with numeric columns."""\n    group = "request_id"\n    exprs = [pl.len().over(group).alias("rq_size"),\n             pl.col(ITEM).n_unique().over(group).alias("rq_items"),\n             pl.len().over([group, ITEM]).alias("rq_item_dup")]\n    for c in RANKED:\n        exprs += [\n            (pl.col(c).rank("average").over(group) / pl.len().over(group)).alias(f"rq_rank_{c}"),\n            (pl.col(c) - pl.col(c).mean().over(group)).alias(f"rq_dev_{c}"),\n            (pl.col(c) - pl.col(c).max().over(group)).alias(f"rq_gap_{c}"),\n        ]\n    out = frame.select("row", *exprs)\n    return out, [c for c in out.columns if c != "row"]\n\n\ndef target_encode(train, valid, column, folds):\n    prior = float(train["watched"].mean())\n    y = train["watched"].to_numpy().astype(np.float64)\n    keys = train[column]\n    oof = np.empty(len(train))\n    support = np.empty(len(train))\n    for fit, apply in folds:\n        stats = (pl.DataFrame({"k": keys[fit], "y": y[fit]})\n                 .group_by("k").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")))\n        joined = pl.DataFrame({"k": keys[apply]}).join(stats, on="k", how="left").fill_null(0)\n        s, n = joined["s"].to_numpy(), joined["n"].to_numpy()\n        oof[apply] = (s + SMOOTHING * prior) / (n + SMOOTHING)\n        support[apply] = n\n    stats = (pl.DataFrame({"k": keys, "y": y})\n             .group_by("k").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")))\n    joined = pl.DataFrame({"k": valid[column]}).join(stats, on="k", how="left").fill_null(0)\n    s, n = joined["s"].to_numpy(), joined["n"].to_numpy()\n    held = (s + SMOOTHING * prior) / (n + SMOOTHING)\n    return (oof, support), (held, n)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t023"))\n    parser.add_argument("--variants", default="base,request,counts,target,all")\n    parser.add_argument("--learning-rate", type=float, default=0.05)\n    parser.add_argument("--rounds", type=int, default=4000)\n    parser.add_argument("--threads", type=int, default=8)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    schema = pl.scan_parquet(args.data / "train.parquet").collect_schema()\n    numeric = scalar_numeric(schema)\n    key_columns = ["request_id", "feature1", *sorted(set(COUNT_KEYS + ENCODED + LOW_CATEGORICAL), key=lambda c: int(c[7:]))]\n    train = pl.scan_parquet(args.data / "train.parquet").with_row_index("row")\n    if args.rows:\n        train = train.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    train = train.select("row", "watched", "feature3", *key_columns, *numeric).collect()\n    test_keys = pl.scan_parquet(args.data / "test.parquet").select(*key_columns)\n    if args.rows:\n        test_keys = test_keys.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    test_keys = test_keys.collect()\n    offset = int(train["row"].max()) + 1\n    keys = pl.concat([\n        train.select("row", *key_columns),\n        test_keys.with_row_index("row", offset=offset).with_columns(pl.col("row").cast(train["row"].dtype)),\n    ])\n    del test_keys\n    counts, count_names = count_features(keys)\n    del keys\n    gc.collect()\n    train = train.join(counts, on="row", how="left")\n    del counts\n    rq, rq_names = request_features(train.select("row", "request_id", ITEM, *RANKED))\n    train = train.join(rq, on="row", how="left")\n    del rq\n    is_valid = (train["feature1"] >= CUTOFF).to_numpy()\n    fit_rows, valid_rows = train.filter(~pl.Series(is_valid)), train.filter(pl.Series(is_valid))\n    del train\n    gc.collect()\n    folds = list(GroupKFold(5).split(np.zeros(len(fit_rows)), groups=fit_rows["request_id"].to_numpy()))\n    te_fit, te_valid, te_names = {}, {}, []\n    for column in ENCODED:\n        (a, an), (b, bn) = target_encode(fit_rows, valid_rows, column, folds)\n        te_fit[f"te_{column}"], te_valid[f"te_{column}"] = a, b\n        te_fit[f"ten_{column}"], te_valid[f"ten_{column}"] = an, bn\n        te_names += [f"te_{column}", f"ten_{column}"]\n    categories = {}\n    for c in LOW_CATEGORICAL:\n        vocabulary = fit_rows[c].drop_nulls().unique().sort()\n        categories[c] = vocabulary\n    base = ["feature3", *numeric, *LOW_CATEGORICAL]\n    groups = {"base": base, "request": base + rq_names, "counts": base + count_names,\n              "target": base + te_names, "all": base + rq_names + count_names + te_names}\n\n    def matrix(frame, te, columns):\n        parts = []\n        for c in columns:\n            if c in te:\n                parts.append(te[c].astype(np.float32))\n            elif c in categories:\n                codes = frame.select(pl.col(c).cast(pl.Enum(categories[c].to_list()), strict=False)\n                                     .to_physical().cast(pl.Float32))[c].to_numpy()\n                parts.append(codes)\n            else:\n                parts.append(frame[c].cast(pl.Float32).to_numpy())\n        return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)\n\n    y_fit = fit_rows["watched"].to_numpy()\n    y_valid = valid_rows["watched"].to_numpy()\n    valid_ids = valid_rows["row"].to_numpy()\n    results = {}\n    for name in args.variants.split(","):\n        columns = groups[name]\n        began = time.time()\n        x_fit = matrix(fit_rows, te_fit, columns)\n        x_valid = matrix(valid_rows, te_valid, columns)\n        categorical = [i for i, c in enumerate(columns) if c in categories]\n        train_set = lgb.Dataset(x_fit, y_fit, feature_name=columns,\n                                categorical_feature=categorical, free_raw_data=True)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        params = {\n            "objective": "binary", "metric": "auc", "learning_rate": args.learning_rate,\n            "num_leaves": 127, "min_data_in_leaf": 200, "feature_fraction": 0.5,\n            "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n            "max_bin": 127, "seed": 42, "num_threads": args.threads, "verbosity": -1,\n            "cat_smooth": 20, "max_cat_to_onehot": 8,\n        }\n        log = {}\n        model = lgb.train(params, train_set, args.rounds, valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.record_evaluation(log),\n                                     lgb.log_evaluation(250)])\n        prediction = model.predict(x_valid, num_iteration=model.best_iteration)\n        auc = float(roc_auc_score(y_valid, prediction))\n        curve = log["holdout"]["auc"]\n        importance = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {\n            "holdout_auc": auc, "best_iteration": model.best_iteration,\n            "auc_at_140": curve[min(139, len(curve) - 1)], "features": len(columns),\n            "seconds": time.time() - began,\n            "top_gain": [[c, float(g)] for c, g in importance],\n        }\n        np.save(args.output / f"valid-{name}.npy", prediction.astype(np.float32))\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        print(" top:", [c for c, _ in importance[:15]], flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)),\n            "fit_rows": int(len(y_fit)), "early_stopping_on_holdout": True,\n            "results": results, "seconds": time.time() - start}, indent=2) + "\\n")\n        del x_fit, x_valid, train_set, valid_set, model\n        gc.collect()\n    np.save(args.output / "valid-row-ids.npy", valid_ids)\n\n\nif __name__ == "__main__":\n    main()\n', 'request_screen.py': '"""T026 local LightGBM screen: wider within-request context features.\n\nSame protocol as T023 (train before the original cutoff, score the original\n307,528-row holdout, early stopping on the holdout for every variant, so deltas\nare comparable). Wide features are chosen by gain from a fixed-budget base\nmodel fitted on training rows only; the holdout is never used to choose them.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\n\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, RANKED, scalar_numeric\n\nPARAMS = {\n    "objective": "binary", "metric": "auc", "num_leaves": 127, "min_data_in_leaf": 200,\n    "feature_fraction": 0.5, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n    "max_bin": 127, "seed": 42, "verbosity": -1, "cat_smooth": 20, "max_cat_to_onehot": 8,\n}\n\n\ndef context(columns, stats=()):\n    group = "request_id"\n    size = pl.len().over(group)\n    exprs = [size.alias("rq_size"), pl.col(ITEM).n_unique().over(group).alias("rq_items"),\n             pl.len().over([group, ITEM]).alias("rq_item_dup")]\n    for c in columns:\n        exprs += [(pl.col(c).rank("average").over(group) / size).alias(f"rq_rank_{c}"),\n                  (pl.col(c) - pl.col(c).mean().over(group)).alias(f"rq_dev_{c}"),\n                  (pl.col(c) - pl.col(c).max().over(group)).alias(f"rq_gap_{c}")]\n    for c in stats:\n        exprs += [pl.col(c).mean().over(group).alias(f"rq_mean_{c}"),\n                  pl.col(c).std().over(group).alias(f"rq_std_{c}")]\n    return [e.cast(pl.Float32) for e in exprs]\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t026"))\n    parser.add_argument("--variants", default="control,no_size,wide,wide_stats")\n    parser.add_argument("--wide", type=int, default=50)\n    parser.add_argument("--threads", type=int, default=6)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    numeric = scalar_numeric(pl.scan_parquet(args.data / "train.parquet").collect_schema())\n    frame = pl.scan_parquet(args.data / "train.parquet")\n    if args.rows:\n        frame = frame.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    frame = frame.select("watched", "feature1", "feature3", "request_id", ITEM,\n                         *dict.fromkeys([*LOW_CATEGORICAL, *numeric])).collect()\n    is_valid = (frame["feature1"] >= CUTOFF).to_numpy()\n    categories = {c: frame.filter(pl.Series(~is_valid))[c].drop_nulls().unique().sort().to_list()\n                  for c in LOW_CATEGORICAL}\n    base = ["feature3", *numeric, *LOW_CATEGORICAL]\n\n    def matrix(source, columns):\n        parts = []\n        for c in columns:\n            column = pl.col(c)\n            if c in categories:\n                column = column.cast(pl.Enum(categories[c]), strict=False).to_physical()\n            parts.append(source.select(column.cast(pl.Float32))[c].to_numpy())\n        return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)\n\n    y = frame["watched"].to_numpy()\n    y_fit, y_valid = y[~is_valid], y[is_valid]\n    categorical = lambda columns: [i for i, c in enumerate(columns) if c in categories]\n    x = matrix(frame.filter(pl.Series(~is_valid)), base)\n    selector = lgb.train({**PARAMS, "learning_rate": .05, "num_threads": args.threads},\n                         lgb.Dataset(x, y_fit, feature_name=base, categorical_feature=categorical(base)), 150)\n    gains = dict(zip(base, selector.feature_importance("gain")))\n    del x, selector\n    gc.collect()\n    candidates = [c for c in numeric if c not in categories and c not in {"feature3"}]\n    wide = sorted(candidates, key=lambda c: -gains[c])[:args.wide]\n    wide = list(dict.fromkeys([*RANKED, *wide]))\n    print(f"wide context over {len(wide)} features; top: {wide[10:20]}", flush=True)\n    frame = frame.with_columns(context(wide, wide[:10]))\n    control = ["rq_size", "rq_items", "rq_item_dup"] + [f"rq_{k}_{c}" for c in RANKED for k in ["rank", "dev", "gap"]]\n    wide_names = ["rq_size", "rq_items", "rq_item_dup"] + [f"rq_{k}_{c}" for c in wide for k in ["rank", "dev", "gap"]]\n    groups = {\n        "control": base + control,\n        "no_size": base + [c for c in control if c != "rq_size"],\n        "wide": base + wide_names,\n        "wide_stats": base + wide_names + [f"rq_{k}_{c}" for c in wide[:10] for k in ["mean", "std"]],\n    }\n    fit_rows, valid_rows = frame.filter(pl.Series(~is_valid)), frame.filter(pl.Series(is_valid))\n    del frame\n    gc.collect()\n    results = {}\n    for name in args.variants.split(","):\n        columns, began = groups[name], time.time()\n        train_set = lgb.Dataset(matrix(fit_rows, columns), y_fit, feature_name=columns,\n                                categorical_feature=categorical(columns), free_raw_data=True)\n        x_valid = matrix(valid_rows, columns)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        model = lgb.train({**PARAMS, "learning_rate": .05, "num_threads": args.threads}, train_set, 4000,\n                          valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)])\n        auc = float(roc_auc_score(y_valid, model.predict(x_valid, num_iteration=model.best_iteration)))\n        top = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {"holdout_auc": auc, "best_iteration": model.best_iteration,\n                         "features": len(columns), "seconds": time.time() - began,\n                         "top_gain": [[c, float(g)] for c, g in top]}\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)), "fit_rows": int(len(y_fit)),\n            "early_stopping_on_holdout": True, "wide_features": wide, "results": results,\n            "seconds": time.time() - start}, indent=2) + "\\n")\n        del train_set, valid_set, x_valid, model\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n', 'classic_screen.py': '"""T029 local screen: classic learners on the T026 wide request-context features.\n\nSame protocol as T026 (fit rows before the original cutoff, score the original\n307,528-row holdout, early stopping on the holdout). The wide feature list is the\none T026 chose from training rows only. Holdout predictions are saved with file\nrow ids so they can be blended with the neural models.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\n\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, scalar_numeric\nfrom request_screen import context\n\n\ndef fit_xgboost(x_fit, y_fit, x_valid, y_valid, categorical, threads):\n    import xgboost as xgb\n\n    fit = xgb.QuantileDMatrix(x_fit, y_fit, max_bin=127)\n    valid = xgb.QuantileDMatrix(x_valid, y_valid, ref=fit, max_bin=127)\n    params = {\n        "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist",\n        "max_depth": 0, "max_leaves": 127, "grow_policy": "lossguide", "min_child_weight": 20,\n        "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.5, "lambda": 10,\n        "nthread": threads, "seed": 42, "max_bin": 127,\n    }\n    model = xgb.train(params, fit, 4000, evals=[(valid, "holdout")],\n                      early_stopping_rounds=200, verbose_eval=250)\n    predictions = model.predict(valid, iteration_range=(0, model.best_iteration + 1))\n    return predictions, {"best_iteration": int(model.best_iteration)}\n\n\ndef fit_catboost(x_fit, y_fit, x_valid, y_valid, categorical, threads):\n    from catboost import CatBoostClassifier, Pool\n\n    def pool(x, y):\n        frame = {f"f{i}": x[:, i] for i in range(x.shape[1])}\n        for i in categorical:\n            frame[f"f{i}"] = np.nan_to_num(x[:, i], nan=-1).astype(np.int32)\n        import pandas as pd\n\n        return Pool(pd.DataFrame(frame), y, cat_features=[f"f{i}" for i in categorical])\n\n    fit, valid = pool(x_fit, y_fit), pool(x_valid, y_valid)\n    model = CatBoostClassifier(\n        iterations=4000, learning_rate=0.08, depth=8, border_count=128, l2_leaf_reg=10,\n        eval_metric="AUC", od_type="Iter", od_wait=200, thread_count=threads,\n        random_seed=42, verbose=250, use_best_model=True,\n    )\n    model.fit(fit, eval_set=valid)\n    return model.predict_proba(valid)[:, 1], {"best_iteration": int(model.get_best_iteration())}\n\n\ndef fit_logreg(x_fit, y_fit, x_valid, y_valid, categorical, threads):\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.preprocessing import OneHotEncoder\n\n    sample = np.sort(np.random.default_rng(42).choice(len(y_fit), min(400_000, len(y_fit)), replace=False))\n    x_fit, y_fit = x_fit[sample], y_fit[sample]\n    dense = [i for i in range(x_fit.shape[1]) if i not in set(categorical)]\n\n    keep = np.isnan(x_fit[:, dense]).any(axis=0)\n\n    def transform(x, stats=None):\n        values = x[:, dense]\n        if stats is None:\n            low, high = np.nanpercentile(values, [0.5, 99.5], axis=0)\n            clipped = np.clip(values, low, high)\n            stats = (low, high, np.nanmean(clipped, axis=0), np.nanstd(clipped, axis=0) + 1e-6)\n        low, high, mean, std = stats\n        missing = np.isnan(values)\n        values = (np.clip(values, low, high) - mean) / std\n        values[missing] = 0\n        return np.hstack([values, missing[:, keep]]).astype(np.float32), stats\n\n    fit_values, stats = transform(x_fit)\n    valid_values, _ = transform(x_valid, stats)\n    encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False, dtype=np.float32)\n    fit_codes = encoder.fit_transform(np.nan_to_num(x_fit[:, categorical], nan=-1))\n    valid_codes = encoder.transform(np.nan_to_num(x_valid[:, categorical], nan=-1))\n    fit_values, valid_values = np.hstack([fit_values, fit_codes]), np.hstack([valid_values, valid_codes])\n    model = LogisticRegression(C=0.1, max_iter=500, tol=1e-5)\n    model.fit(fit_values, y_fit)\n    return model.predict_proba(valid_values)[:, 1], {"iterations": int(model.n_iter_[0]),\n                                                    "inputs": int(fit_values.shape[1])}\n\n\ndef build_matrices(data, wide, rows=0):\n    """Fit/holdout float32 matrices of the T029 wide feature set (categoricals as fit-period codes)."""\n    numeric = scalar_numeric(pl.scan_parquet(data / "train.parquet").collect_schema())\n    frame = pl.scan_parquet(data / "train.parquet").with_row_index("_row_id")\n    if rows:\n        frame = frame.filter(pl.col("request_id").hash(7) % rows == 0)\n    frame = frame.select("_row_id", "watched", "feature1", "feature3", "request_id", ITEM,\n                         *dict.fromkeys([*LOW_CATEGORICAL, *numeric])).collect()\n    frame = frame.with_columns(context(wide))\n    is_valid = (frame["feature1"] >= CUTOFF).to_numpy()\n    categories = {c: frame.filter(pl.Series(~is_valid))[c].drop_nulls().unique().sort().to_list()\n                  for c in LOW_CATEGORICAL}\n    columns = ["feature3", *numeric, *LOW_CATEGORICAL, "rq_size", "rq_items", "rq_item_dup",\n               *[f"rq_{k}_{c}" for c in wide for k in ["rank", "dev", "gap"]]]\n    categorical = [i for i, c in enumerate(columns) if c in categories]\n\n    y = frame["watched"].to_numpy()\n    y_fit, y_valid = y[~is_valid], y[is_valid]\n    row_ids = frame["_row_id"].to_numpy()[is_valid]\n    x_fit = np.empty((int((~is_valid).sum()), len(columns)), dtype=np.float32)\n    x_valid = np.empty((int(is_valid.sum()), len(columns)), dtype=np.float32)\n    for i, c in enumerate(columns):\n        column = pl.col(c)\n        if c in categories:\n            column = column.cast(pl.Enum(categories[c]), strict=False).to_physical()\n        values = frame.select(column.cast(pl.Float32))[c].to_numpy()\n        x_fit[:, i], x_valid[:, i] = values[~is_valid], values[is_valid]\n\n    del frame\n    gc.collect()\n    return x_fit, y_fit, x_valid, y_valid, row_ids, columns, categorical\n\n\nMODELS = {"xgboost": fit_xgboost, "catboost": fit_catboost, "logreg": fit_logreg}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--wide-from", type=Path, default=Path("artifacts/local-t026/results.json"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t029"))\n    parser.add_argument("--models", default="xgboost,logreg,catboost")\n    parser.add_argument("--threads", type=int, default=7)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    wide = json.loads(args.wide_from.read_text())["wide_features"]\n    x_fit, y_fit, x_valid, y_valid, row_ids, columns, categorical = build_matrices(args.data, wide, args.rows)\n    print(f"matrices {x_fit.shape} / {x_valid.shape} in {time.time() - start:.0f}s", flush=True)\n    path = args.output / "results.json"\n    results = json.loads(path.read_text())["results"] if path.exists() else {}\n    for name in args.models.split(","):\n        began = time.time()\n        predictions, extra = MODELS[name](x_fit, y_fit, x_valid, y_valid, categorical, args.threads)\n        auc = float(roc_auc_score(y_valid, predictions))\n        np.savez_compressed(args.output / f"{name}-holdout.npz", row_ids=row_ids,\n                            y_true=y_valid, probabilities=predictions.astype(np.float32))\n        results[name] = {"holdout_auc": auc, "seconds": time.time() - began, **extra}\n        print(name, json.dumps(results[name]), flush=True)\n        path.write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)), "fit_rows": int(len(y_fit)),\n            "features": len(columns), "early_stopping_on_holdout": True, "wide_features": wide,\n            "results": results}, indent=2) + "\\n")\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n', 'xgb_tune.py': '"""T034: XGBoost GPU tuning on the T029 wide request-context features.\n\n`build` writes the fit/holdout matrices once as .npy; `worker` fits a list of named\nconfigs (one process per GPU) and saves holdout predictions with file row ids.\nProtocol matches T029: original cutoff, early stopping on the holdout.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nfrom sklearn.metrics import roc_auc_score\n\nBASE = {\n    "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "device": "cuda",\n    "max_depth": 0, "max_leaves": 127, "grow_policy": "lossguide", "min_child_weight": 20,\n    "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.5, "lambda": 10, "seed": 42, "max_bin": 127,\n}\n\nCONFIGS = {\n    "t029_gpu": {},\n    "leaves255_eta02": {"max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 100},\n    "leaves511_eta02": {"max_leaves": 511, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 200,\n                        "alpha": 1},\n    "leaves63_eta02": {"max_leaves": 63, "eta": 0.02, "colsample_bytree": 0.4, "min_child_weight": 50},\n    "depth10": {"grow_policy": "depthwise", "max_depth": 10, "max_leaves": 0, "eta": 0.03,\n                "colsample_bytree": 0.4, "min_child_weight": 50},\n    "depth12_node": {"grow_policy": "depthwise", "max_depth": 12, "max_leaves": 0, "eta": 0.02,\n                     "colsample_bytree": 0.8, "colsample_bynode": 0.4, "min_child_weight": 100, "gamma": 1},\n    "bin255": {"max_bin": 255, "max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3,\n               "min_child_weight": 100},\n    "native_cat": {"max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 100,\n                   "max_cat_to_onehot": 8, "max_cat_threshold": 64, "_categorical": True},\n    "subsample06": {"max_leaves": 255, "eta": 0.02, "subsample": 0.6, "colsample_bytree": 0.3,\n                    "min_child_weight": 100, "sampling_method": "gradient_based"},\n    "lambda50": {"max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 100,\n                 "lambda": 50, "alpha": 5},\n}\n\n\ndef build(args):\n    from classic_screen import build_matrices\n\n    start = time.time()\n    wide = json.loads(args.wide_from.read_text())["wide_features"]\n    x_fit, y_fit, x_valid, y_valid, row_ids, columns, categorical = build_matrices(args.data, wide, args.rows)\n    args.matrices.mkdir(parents=True, exist_ok=True)\n    for name, value in dict(x_fit=x_fit, y_fit=y_fit, x_valid=x_valid, y_valid=y_valid, row_ids=row_ids).items():\n        np.save(args.matrices / f"{name}.npy", value)\n    (args.matrices / "columns.json").write_text(json.dumps({"columns": columns, "categorical": categorical}))\n    print(f"matrices {x_fit.shape} / {x_valid.shape} in {time.time() - start:.0f}s", flush=True)\n\n\ndef worker(args):\n    import xgboost as xgb\n\n    m = args.matrices\n    x_fit, x_valid = np.load(m / "x_fit.npy", mmap_mode="r"), np.load(m / "x_valid.npy", mmap_mode="r")\n    y_fit, y_valid, row_ids = np.load(m / "y_fit.npy"), np.load(m / "y_valid.npy"), np.load(m / "row_ids.npy")\n    meta = json.loads((m / "columns.json").read_text())\n    feature_types = ["c" if i in set(meta["categorical"]) else "q" for i in range(len(meta["columns"]))]\n    args.output.mkdir(parents=True, exist_ok=True)\n    for name in args.configs.split(","):\n        began = time.time()\n        params = {**BASE, **CONFIGS[name], "device": args.device}\n        categorical = params.pop("_categorical", False)\n        types = dict(feature_types=feature_types, enable_categorical=True) if categorical else {}\n        fit = xgb.QuantileDMatrix(np.asarray(x_fit), y_fit, max_bin=params["max_bin"], **types)\n        valid = xgb.QuantileDMatrix(np.asarray(x_valid), y_valid, ref=fit, max_bin=params["max_bin"], **types)\n        model = xgb.train(params, fit, args.rounds, evals=[(valid, "holdout")],\n                          early_stopping_rounds=300, verbose_eval=500)\n        predictions = model.predict(valid, iteration_range=(0, model.best_iteration + 1))\n        auc = float(roc_auc_score(y_valid, predictions))\n        np.savez_compressed(args.output / f"{name}-holdout.npz", row_ids=row_ids, y_true=y_valid,\n                            probabilities=predictions.astype(np.float32))\n        gain = model.get_score(importance_type="total_gain")\n        top = sorted(gain.items(), key=lambda kv: -kv[1])[:30]\n        result = {"holdout_auc": auc, "best_iteration": int(model.best_iteration),\n                  "seconds": time.time() - began, "params": params,\n                  "top_gain": [[meta["columns"][int(k[1:])], v] for k, v in top]}\n        (args.output / f"{name}.json").write_text(json.dumps(result, indent=2) + "\\n")\n        print(name, json.dumps({k: result[k] for k in ["holdout_auc", "best_iteration", "seconds"]}), flush=True)\n        del fit, valid, model\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("mode", choices=["build", "worker"])\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--wide-from", type=Path, default=Path("artifacts/local-t026/results.json"))\n    parser.add_argument("--matrices", type=Path, default=Path("artifacts/local-t034/matrices"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t034"))\n    parser.add_argument("--configs", default=",".join(CONFIGS))\n    parser.add_argument("--device", default="cuda")\n    parser.add_argument("--rounds", type=int, default=20000)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    build(args) if args.mode == "build" else worker(args)\n\n\nif __name__ == "__main__":\n    main()\n'}
frozen = Path("/kaggle/working/frozen-source")
frozen.mkdir()
for name, source in sources.items():
    (frozen / name).write_text(source)
(frozen / "t026-wide.json").write_text('{"wide_features": ["feature418", "feature419", "feature411", "feature410", "feature430", "feature431", "feature417", "feature352", "feature346", "feature480", "feature54", "feature493", "feature358", "feature515", "feature490", "feature429", "feature492", "feature510", "feature334", "feature511", "feature103", "feature39", "feature416", "feature484", "feature503", "feature150", "feature356", "feature151", "feature53", "feature337", "feature512", "feature478", "feature428", "feature514", "feature37", "feature513", "feature357", "feature376", "feature502", "feature439", "feature479", "feature507", "feature487", "feature333", "feature50", "feature320", "feature117", "feature126", "feature491", "feature56", "feature153"]}')
data = [p.parent for p in Path("/kaggle/input").rglob("train.parquet") if (p.parent / "sample_submission.csv").is_file()]
if len(data) != 1:
    raise RuntimeError(f"Expected competition data: {data}")
import xgboost
gpus = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 1)
print("xgboost", xgboost.__version__, "cpus", os.cpu_count(), "gpus", gpus, flush=True)
subprocess.run("nvidia-smi; free -m", shell=True, check=False)
monitor = subprocess.Popen("while true; do echo MEM $(date +%T) $(free -m | sed -n 2p); nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader | tr '\n' ' '; echo; sleep 120; done", shell=True)
matrices = "/tmp/t034-matrices"
try:
    subprocess.run([sys.executable, str(frozen / "xgb_tune.py"), "build", "--data", str(data[0]),
                    "--wide-from", str(frozen / "t026-wide.json"), "--matrices", matrices], check=True, cwd=frozen)
    configs = ['t029_gpu', 'leaves255_eta02', 'leaves511_eta02', 'leaves63_eta02', 'depth10', 'depth12_node', 'bin255', 'native_cat', 'subsample06', 'lambda50']
    workers = []
    for gpu in range(gpus):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        log = open(f"/kaggle/working/worker{gpu}.log", "w")
        workers.append((subprocess.Popen([sys.executable, str(frozen / "xgb_tune.py"), "worker",
                                          "--matrices", matrices, "--output", "/kaggle/working/t034",
                                          "--configs", ",".join(configs[gpu::gpus])],
                                         cwd=frozen, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    codes = [p.wait() for p, _ in workers]
    for gpu in range(gpus):
        print(f"===== worker{gpu} exit {codes[gpu]}", flush=True)
        print("\n".join(l for l in Path(f"/kaggle/working/worker{gpu}.log").read_text().splitlines()
                         if "WARNING" not in l), flush=True)
    if any(codes):
        raise RuntimeError(f"worker exit codes {codes}")
finally:
    monitor.kill()
